# 🇬🇧➡️🇵🇰 English to Urdu Neural Machine Translation using LSTM

This notebook builds a **Sequence-to-Sequence (Seq2Seq) Encoder–Decoder** translator that converts **English sentences into Urdu**, using a **LSTM** recurrent neural network.

### What you'll learn in this notebook
1. How text is cleaned and prepared for translation models
2. How to tokenize and pad two different languages
3. How an **Encoder–Decoder LSTM architecture** works for translation
4. How to train the model and visualize its learning
5. How to build an **inference pipeline** to translate new sentences

> 💡 **Note:** This notebook ships with a small built-in sample English–Urdu dataset so it runs out-of-the-box on Kaggle.
> For real results, attach a larger parallel corpus (e.g. an English–Urdu dataset from Kaggle) and point `DATA_PATH` to it — the loading cell below will automatically use it if found.


## 1. Import Libraries

We use **TensorFlow / Keras** to build the neural network, **NumPy/Pandas** for data handling, and **Matplotlib** to visualize training progress.

In [ ]:
# ===== Core libraries =====
import os
import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ===== Deep learning libraries (TensorFlow / Keras) =====
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# Make results reproducible
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)


## 2. Load the Dataset

We expect a CSV file with two columns: **`english`** and **`urdu`** (parallel sentence pairs).

- On Kaggle, upload/attach an English–Urdu parallel corpus and set `DATA_PATH` below to its location (usually something like `/kaggle/input/<dataset-name>/<file>.csv`).
- If no such file is found, we fall back to a **small built-in sample dataset** so the notebook still runs end-to-end (great for understanding the pipeline, but too small to produce great translations).

In [ ]:
# ===== Path to your dataset (change this to your Kaggle dataset path) =====
DATA_PATH = "/kaggle/input/english-to-urdu/english_to_urdu.csv"  # <-- update this if needed

# A tiny built-in fallback dataset (English -> Urdu) so the notebook always runs
sample_data = {
    "english": [
        "hello", "how are you", "what is your name", "thank you", "good morning",
        "good night", "i am fine", "where are you going", "i love you", "what time is it",
        "how much does this cost", "i am hungry", "please help me", "see you tomorrow",
        "what is this", "i am happy", "i am sad", "come here", "go away", "welcome",
        "nice to meet you", "have a nice day", "i am sorry", "excuse me", "how old are you",
        "i am tired", "let us go", "wait for me", "call me later", "i understand"
    ],
    "urdu": [
        "ہیلو", "آپ کیسے ہیں", "آپ کا نام کیا ہے", "شکریہ", "صبح بخیر",
        "شب بخیر", "میں ٹھیک ہوں", "آپ کہاں جا رہے ہیں", "میں آپ سے محبت کرتا ہوں", "کیا وقت ہوا ہے",
        "اس کی قیمت کتنی ہے", "مجھے بھوک لگی ہے", "میری مدد کریں", "کل ملتے ہیں",
        "یہ کیا ہے", "میں خوش ہوں", "میں اداس ہوں", "یہاں آؤ", "چلے جاؤ", "خوش آمدید",
        "آپ سے مل کر خوشی ہوئی", "آپ کا دن اچھا گزرے", "معاف کیجئے", "معذرت",
        "آپ کی عمر کیا ہے", "میں تھک گیا ہوں", "چلو چلتے ہیں", "میرا انتظار کرو",
        "مجھے بعد میں کال کریں", "میں سمجھ گیا"
    ]
}

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    # Keep only the two columns we need, drop empty rows
    df = df[["english", "urdu"]].dropna().reset_index(drop=True)
    print(f"Loaded real dataset from {DATA_PATH} with {len(df)} sentence pairs.")
else:
    df = pd.DataFrame(sample_data)
    print("DATA_PATH not found -> using the small built-in sample dataset "
          f"with {len(df)} sentence pairs (for demonstration only).")

df.head()


## 3. Text Cleaning & Preprocessing

Before feeding text to a neural network we need to:
1. Lowercase English text and strip punctuation (Urdu punctuation is handled separately since Urdu script has its own characters).
2. Add special **`<start>`** and **`<end>`** tokens to every Urdu sentence, so the decoder knows when translation should begin and stop.


In [ ]:
def clean_english(text):
    """Lowercase, remove punctuation/extra spaces from an English sentence."""
    text = str(text).lower().strip()
    text = re.sub(r"[" + re.escape(string.punctuation) + r"]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_urdu(text):
    """
    Clean an Urdu sentence and wrap it with <start> / <end> tokens.
    These tokens tell the decoder where a sentence begins and ends.
    """
    text = str(text).strip()
    # Remove common punctuation marks (Urdu uses '۔' as a full stop, keep letters/spaces)
    text = re.sub(r"[۔،؟!.,?]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return "<start> " + text + " <end>"


# Apply cleaning to the whole dataset
df["english_clean"] = df["english"].apply(clean_english)
df["urdu_clean"] = df["urdu"].apply(clean_urdu)

df[["english_clean", "urdu_clean"]].head()


## 4. Tokenization & Padding

Neural networks work with **numbers**, not words. So we:
1. Build a **vocabulary** (word → integer id) separately for English and Urdu using Keras' `Tokenizer`.
2. Convert every sentence into a sequence of integers.
3. **Pad** all sequences to the same length (shorter sentences are padded with zeros) so they can be processed in batches.

In [ ]:
# ===== English tokenizer (encoder input) =====
eng_tokenizer = Tokenizer(filters="")
eng_tokenizer.fit_on_texts(df["english_clean"])
eng_vocab_size = len(eng_tokenizer.word_index) + 1  # +1 for padding token (index 0)

eng_sequences = eng_tokenizer.texts_to_sequences(df["english_clean"])
max_eng_len = max(len(seq) for seq in eng_sequences)

encoder_input_data = pad_sequences(eng_sequences, maxlen=max_eng_len, padding="post")

# ===== Urdu tokenizer (decoder input/output) =====
urdu_tokenizer = Tokenizer(filters="")
urdu_tokenizer.fit_on_texts(df["urdu_clean"])
urdu_vocab_size = len(urdu_tokenizer.word_index) + 1

urdu_sequences = urdu_tokenizer.texts_to_sequences(df["urdu_clean"])
max_urdu_len = max(len(seq) for seq in urdu_sequences)

decoder_full_data = pad_sequences(urdu_sequences, maxlen=max_urdu_len, padding="post")

print("English vocabulary size:", eng_vocab_size)
print("Urdu vocabulary size:", urdu_vocab_size)
print("Max English sentence length:", max_eng_len)
print("Max Urdu sentence length:", max_urdu_len)


### Decoder Input vs Decoder Target (Teacher Forcing)

In Seq2Seq training we use **teacher forcing**:
- **Decoder input** = Urdu sentence *without* the last token (starts with `<start>`)
- **Decoder target** = Urdu sentence *without* the first token (ends with `<end>`), one-hot encoded

This trains the decoder to predict the **next word**, given the correct previous word.

In [ ]:
# Decoder input: everything except the last time-step
decoder_input_data = decoder_full_data[:, :-1]
# Decoder target: everything except the first time-step (shifted by one)
decoder_target_data = decoder_full_data[:, 1:]

# One-hot encode the decoder targets (needed for categorical_crossentropy)
decoder_target_onehot = to_categorical(decoder_target_data, num_classes=urdu_vocab_size)

print("Encoder input shape:", encoder_input_data.shape)
print("Decoder input shape:", decoder_input_data.shape)
print("Decoder target (one-hot) shape:", decoder_target_onehot.shape)


## 5. Train / Validation Split

We split the data so we can check how well the model generalizes to sentences it wasn't trained on.

In [ ]:
(enc_train, enc_val,
 dec_in_train, dec_in_val,
 dec_out_train, dec_out_val) = train_test_split(
    encoder_input_data,
    decoder_input_data,
    decoder_target_onehot,
    test_size=0.2,
    random_state=42
)

print("Training samples:", enc_train.shape[0])
print("Validation samples:", enc_val.shape[0])


## 6. Build the Encoder–Decoder LSTM Model

**How Seq2Seq translation works:**

- The **Encoder** reads the English sentence word-by-word and compresses its meaning into a fixed-size "context" (the final hidden states of the LSTM).
- The **Decoder** takes that context and generates the Urdu sentence one word at a time, using the previous word as input at each step.

An LSTM has two internal states — a **hidden state** and a **cell state** — which together help it remember information over longer sentences.

```
English sentence  ---> [Embedding] ---> [LSTM Encoder] ---> context vector
                                                                        |
                                                                        v
<start> Urdu ...  ---> [Embedding] ---> [LSTM Decoder] ---> [Dense+Softmax] ---> next Urdu word
```


In [ ]:
# ===== Hyperparameters =====
EMBEDDING_DIM = 256   # size of word embedding vectors
LATENT_DIM = 256       # number of units in the LSTM layer

# ===================== ENCODER =====================
encoder_inputs = Input(shape=(max_eng_len,), name="encoder_inputs")
encoder_embedding = Embedding(
    input_dim=eng_vocab_size,
    output_dim=EMBEDDING_DIM,
    mask_zero=True,          # ignore padding (zeros) during training
    name="encoder_embedding"
)(encoder_inputs)

# LSTM has TWO internal states: hidden state (h) and cell state (c)
encoder_lstm = LSTM(LATENT_DIM, return_state=True, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)
encoder_states = [state_h, state_c]  # this is the "context" passed to the decoder

# ===================== DECODER =====================
decoder_inputs = Input(shape=(max_urdu_len - 1,), name="decoder_inputs")
decoder_embedding_layer = Embedding(
    input_dim=urdu_vocab_size,
    output_dim=EMBEDDING_DIM,
    mask_zero=True,
    name="decoder_embedding"
)
decoder_embedding = decoder_embedding_layer(decoder_inputs)

# The decoder is initialized with the encoder's final state (the "context")
decoder_lstm = LSTM(LATENT_DIM, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=encoder_states)

# Dense + Softmax turns the decoder's output into a probability distribution over the Urdu vocabulary
decoder_dense = Dense(urdu_vocab_size, activation="softmax", name="decoder_dense")
decoder_outputs = decoder_dense(decoder_outputs)

# ===================== FULL TRAINING MODEL =====================
model = Model([encoder_inputs, decoder_inputs], decoder_outputs, name=f"eng_to_urdu_lstm")
model.summary()


## 7. Compile & Train the Model

- **Loss:** `categorical_crossentropy` — since we predict a probability distribution over the Urdu vocabulary at every time step.
- **Optimizer:** `adam` — a good general-purpose optimizer for sequence models.
- **Metric:** `accuracy` — how many words are predicted correctly (a rough guide, not a perfect translation-quality metric).

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

EPOCHS = 100          # increase this a lot (e.g. 300+) once you use a real, larger dataset
BATCH_SIZE = 16

history = model.fit(
    [enc_train, dec_in_train],
    dec_out_train,
    validation_data=([enc_val, dec_in_val], dec_out_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    verbose=1
)


## 8. Visualize Training Progress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ---- Loss curve ----
axes[0].plot(history.history["loss"], label="Train Loss")
axes[0].plot(history.history["val_loss"], label="Validation Loss")
axes[0].set_title(f"LSTM Model - Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

# ---- Accuracy curve ----
axes[1].plot(history.history["accuracy"], label="Train Accuracy")
axes[1].plot(history.history["val_accuracy"], label="Validation Accuracy")
axes[1].set_title(f"LSTM Model - Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()


## 9. Build Inference (Translation) Models

During **training**, the decoder sees the *entire correct* Urdu sentence at once (teacher forcing).
During **inference** (real translation), we don't know the Urdu sentence in advance — so we must generate it **one word at a time**:

1. Encode the English sentence to get the context (hidden + cell state).
2. Feed `<start>` into the decoder together with that context to predict the first Urdu word.
3. Feed the predicted word back into the decoder to predict the next word.
4. Repeat until the model predicts `<end>` or we hit the maximum length.

For this we build two small helper models re-using the **same trained layers**.

In [ ]:
# ===== Encoder inference model: English sentence -> context state(s) =====
encoder_model_inf = Model(encoder_inputs, encoder_states, name="encoder_inference")

# ===== Decoder inference model: one word + previous state -> next word + new state =====
decoder_state_input_h = Input(shape=(LATENT_DIM,), name="decoder_state_input_h")
decoder_state_input_c = Input(shape=(LATENT_DIM,), name="decoder_state_input_c")
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_inputs_inf = Input(shape=(1,), name="decoder_inputs_inf")  # feed ONE token at a time
decoder_embedding_inf = decoder_embedding_layer(decoder_inputs_inf)

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_embedding_inf, initial_state=decoder_states_inputs
)
decoder_states_inf = [state_h_inf, state_c_inf]

decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model_inf = Model(
    [decoder_inputs_inf] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf,
    name="decoder_inference"
)

print("Inference models ready.")


## 10. Translation Function

This function ties the encoder and decoder inference models together to translate a **new English sentence** into Urdu, word by word.

In [ ]:
urdu_index_to_word = {idx: word for word, idx in urdu_tokenizer.word_index.items()}
# NOTE: because we used Tokenizer(filters="") the angle brackets are kept, so the
# tokens are literally "<start>" and "<end>" (not "start"/"end").
start_token_id = urdu_tokenizer.word_index["<start>"]
end_token_id = urdu_tokenizer.word_index["<end>"]


def translate_sentence(english_sentence, max_output_len=max_urdu_len):
    """Translate a single English sentence into Urdu using greedy decoding."""
    # ---- Preprocess & encode the English input exactly like during training ----
    cleaned = clean_english(english_sentence)
    seq = eng_tokenizer.texts_to_sequences([cleaned])
    input_seq = pad_sequences(seq, maxlen=max_eng_len, padding="post")

    # 1. Encode the input sentence into its context state (h, c)
    states_value = encoder_model_inf.predict(input_seq, verbose=0)

    # 2. Start the decoder with the <start> token
    target_seq = np.array([[start_token_id]])

    translated_words = []
    for _ in range(max_output_len):
        output_tokens, h, c = decoder_model_inf.predict(
            [target_seq] + states_value, verbose=0
        )
        states_value = [h, c]

        # 3. Pick the most probable next word (greedy decoding)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])

        if sampled_token_index == 0:  # padding index, stop
            break

        sampled_word = urdu_index_to_word.get(sampled_token_index, "")

        if sampled_word == "<end>" or sampled_token_index == end_token_id:
            break

        translated_words.append(sampled_word)

        # 4. Feed the predicted word back in as input for the next step
        target_seq = np.array([[sampled_token_index]])

    return " ".join(translated_words)


## 11. Try It Out! 🎉

Let's translate a few English sentences (including some from our training data, and a new one) into Urdu.

In [ ]:
test_sentences = [
    "hello",
    "thank you",
    "how are you",
    "i am hungry",
    "good morning"
]

for sentence in test_sentences:
    translation = translate_sentence(sentence)
    print(f"EN: {sentence}")
    print(f"UR: {translation}")
    print("-" * 40)


## 12. Save the Trained Model (Optional)

So you can reload it later without retraining.

In [ ]:
model.save(f"english_to_urdu_lstm.keras")
print("Model saved!")


## 13. Conclusion & Next Steps

You just built a full **English → Urdu Encoder–Decoder translator using LSTM** from scratch:
data cleaning → tokenization → model building → training → inference → translation. 🎉

**Ways to improve this notebook:**
- Use a **much larger** real parallel English–Urdu dataset (this notebook's built-in sample is tiny and only for demonstration).
- Add an **Attention mechanism** so the decoder can "look back" at all encoder outputs instead of a single context vector — this greatly improves translation quality for longer sentences.
- Try **Bidirectional** encoder layers.
- Compare this LSTM model against the companion **GRU** notebook to see which architecture performs better on your dataset.
- Use **BLEU score** instead of accuracy to properly evaluate translation quality.

Happy translating! 🚀
